# Time Series Statistics

## Learning Objectives
1. Understand AR(1) processes, the stationarity condition |phi| < 1, and how the ACF decays
2. Implement stationarity testing and differencing using ADF-equivalent diagnostics with numpy
3. Build and evaluate an ARIMA-style forecasting pipeline with temporal cross-validation
4. Compare ARIMA-style forecasts to naive baselines and assess seasonality decomposition

In [ ]:
# Cell 2: Imports and reproducibility
# Note: statsmodels is not available; we use numpy/scipy implementations throughout
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

np.random.seed(42)

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded (numpy-based time series implementation)')


def compute_acf(x: np.ndarray, max_lag: int = 40) -> np.ndarray:
    """Compute sample autocorrelation function up to max_lag."""
    n = len(x)
    xm = x - x.mean()
    var = np.var(xm)
    if var < 1e-12:
        return np.zeros(max_lag + 1)
    acf = np.array([np.mean(xm[:n-k] * xm[k:]) / var for k in range(max_lag + 1)])
    return acf


def compute_pacf(x: np.ndarray, max_lag: int = 20) -> np.ndarray:
    """Compute partial autocorrelation function using Yule-Walker equations."""
    n = len(x)
    xm = x - x.mean()
    pacf = np.zeros(max_lag + 1)
    pacf[0] = 1.0
    acf = compute_acf(x, max_lag)
    for k in range(1, max_lag + 1):
        # Yule-Walker: R @ phi = r
        R = np.array([[acf[abs(i - j)] for j in range(k)] for i in range(k)])
        r = acf[1:k + 1]
        try:
            phi = np.linalg.solve(R, r)
            pacf[k] = phi[-1]
        except np.linalg.LinAlgError:
            pacf[k] = 0.0
    return pacf


print('ACF and PACF utilities defined.')

## Level 1: AR(1) Process — Simulation and Stationarity Condition

An AR(1) process is: x_t = phi * x_{t-1} + eps_t, eps_t ~ N(0, sigma^2). Stationary iff |phi| < 1. The ACF decays geometrically as phi^k.

In [ ]:
# Cell 4: AR(1) process simulation and ACF analysis

def simulate_ar1(phi: float, sigma: float, n: int, x0: float = 0.0, rng=None) -> np.ndarray:
    """Simulate an AR(1) process: x_t = phi * x_{t-1} + eps_t.
    
    Stationary when |phi| < 1. Long-run mean = 0, variance = sigma^2 / (1 - phi^2).
    The ACF at lag k = phi^k: geometric decay for |phi| < 1.
    """
    if rng is None:
        rng = np.random.default_rng(42)
    x = np.empty(n)
    x[0] = x0
    noise = rng.normal(0, sigma, n)
    for t in range(1, n):
        x[t] = phi * x[t - 1] + noise[t]
    return x


rng = np.random.default_rng(42)
N_TS = 500
phi_values = [0.9, 0.5, -0.7, 1.0]  # Last is non-stationary (random walk)
labels = ['phi=0.9 (stat)', 'phi=0.5 (stat)', 'phi=-0.7 (stat)', 'phi=1.0 (non-stat, RW)']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for col, (phi, label) in enumerate(zip(phi_values, labels)):
    x = simulate_ar1(phi, sigma=1.0, n=N_TS, rng=rng)
    acf_vals = compute_acf(x, max_lag=30)
    # Theoretical ACF for comparison
    lags = np.arange(31)
    theo_acf = phi**lags if abs(phi) < 1 else np.ones(31)  # RW ACF ≈ 1

    # Time series plot
    axes[0, col].plot(x, color='navy', lw=0.8, alpha=0.7)
    axes[0, col].set_title(label, fontsize=9)
    axes[0, col].set_xlabel('t')
    if col == 0:
        axes[0, col].set_ylabel('x_t')

    # ACF plot with theoretical overlay
    axes[1, col].bar(lags, acf_vals, color='steelblue', alpha=0.6, width=0.8, label='Sample ACF')
    axes[1, col].plot(lags, theo_acf, 'r-', lw=2, label=f'Theo: phi^k')
    # 95% CI for white noise: ±1.96/sqrt(N)
    ci = 1.96 / np.sqrt(N_TS)
    axes[1, col].axhline(ci, color='gray', ls='--', lw=1)
    axes[1, col].axhline(-ci, color='gray', ls='--', lw=1)
    axes[1, col].set_xlabel('Lag')
    if col == 0:
        axes[1, col].set_ylabel('ACF')
    axes[1, col].legend(fontsize=7)

plt.suptitle('AR(1) Processes: Time Series and ACF', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ar1_processes.png', dpi=80, bbox_inches='tight')
plt.show()

# Verify theoretical long-run variance for stationary cases
phi_stat = 0.9
x_stat = simulate_ar1(phi_stat, sigma=1.0, n=50_000, rng=rng)
theoretical_var = 1.0 / (1 - phi_stat**2)
print(f'AR(1) phi={phi_stat}: Theoretical var = {theoretical_var:.4f}, Empirical var = {x_stat.var():.4f}')
print('Level 1 complete.')

## Level 2: Stationarity Testing and Differencing

We implement a simple stationarity diagnostic (rolling mean/variance and an AR-coefficient test) as a numpy-based alternative to the ADF test, and demonstrate differencing to achieve stationarity.

In [ ]:
# Cell 6: Stationarity diagnostics and differencing
# Since statsmodels is not available, we implement:
# 1. Rolling mean and rolling variance diagnostic (visual + quantitative)
# 2. AR(1) coefficient test: fit phi by OLS; if phi close to 1, non-stationary
# 3. First differencing to achieve stationarity

def stationarity_diagnostic(x: np.ndarray, window: int = None, label: str = '') -> dict:
    """Compute stationarity diagnostics without statsmodels.
    
    Checks rolling mean stability and AR(1) coefficient as proxy for unit root.
    AR(1) OLS phi estimate near 1 signals non-stationarity.
    
    Returns dict with phi_hat, rolling_mean_std, rolling_var_std
    """
    n = len(x)
    if window is None:
        window = max(n // 5, 20)

    # Rolling mean and variance
    rolling_means = np.array([x[i:i+window].mean() for i in range(n - window + 1)])
    rolling_vars = np.array([x[i:i+window].var() for i in range(n - window + 1)])

    # AR(1) coefficient via OLS: regress x_t on x_{t-1}
    # Phi close to 1 => unit root (non-stationary)
    x_lag = x[:-1].reshape(-1, 1)
    x_curr = x[1:]
    # OLS: phi = (x_lag^T x_curr) / (x_lag^T x_lag)
    phi_hat = float(x_lag.T @ x_curr / (x_lag.T @ x_lag))

    # Standard error of phi_hat under H0: phi=1
    residuals = x_curr - phi_hat * x_lag.ravel()
    sigma2 = residuals.var()
    se_phi = np.sqrt(sigma2 / float(x_lag.T @ x_lag))
    # t-statistic for H0: phi = 1 (not 0, like a unit root test)
    t_stat = (phi_hat - 1.0) / (se_phi + 1e-12)
    # Critical value for unit root: approximately -1.95 (simplified, one-sided)
    # (True Dickey-Fuller critical values differ; this is a rough analog)
    likely_stationary = t_stat < -1.95

    result = {
        'phi_hat': phi_hat,
        't_stat': t_stat,
        'likely_stationary': likely_stationary,
        'rolling_mean_std': rolling_means.std(),
        'rolling_var_std': rolling_vars.std(),
        'rolling_means': rolling_means,
        'rolling_vars': rolling_vars,
    }

    print(f'\n{label}:')
    print(f'  AR(1) phi_hat = {phi_hat:.4f}, t-stat(phi=1) = {t_stat:.3f}')
    print(f'  Rolling mean std = {rolling_means.std():.4f}')
    print(f'  Rolling var  std = {rolling_vars.std():.4f}')
    print(f'  Likely stationary: {likely_stationary}')
    return result


rng = np.random.default_rng(5)
N = 400

# Generate various series for testing
random_walk = np.cumsum(rng.normal(0, 1, N))  # Non-stationary: phi=1
ar1_stat = simulate_ar1(0.7, 1.0, N, rng=rng)  # Stationary
trended = np.arange(N) * 0.3 + rng.normal(0, 2, N)  # Trend non-stationary

# First differences
rw_diff = np.diff(random_walk)  # d=1 differencing
trended_diff = np.diff(trended)

print('=== Stationarity Diagnostics ===')
diag_rw = stationarity_diagnostic(random_walk, label='Random Walk (non-stationary)')
diag_ar1 = stationarity_diagnostic(ar1_stat, label='AR(1) phi=0.7 (stationary)')
diag_trend = stationarity_diagnostic(trended, label='Trended series (non-stationary)')
diag_rw_diff = stationarity_diagnostic(rw_diff, label='Random Walk after diff (stationary?)')
diag_trend_diff = stationarity_diagnostic(trended_diff, label='Trended after diff (stationary?)')

# Plot differencing effect
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for col, (orig, diffed, title) in enumerate([
    (random_walk, rw_diff, 'Random Walk'),
    (ar1_stat, np.diff(ar1_stat), 'AR(1) phi=0.7'),
    (trended, trended_diff, 'Trended Series'),
]):
    axes[0, col].plot(orig, color='firebrick', lw=0.9)
    axes[0, col].set_title(f'Original: {title}')
    axes[0, col].set_ylabel('Value')
    # Rolling mean overlay
    w = 50
    rm = np.array([orig[i:i+w].mean() for i in range(len(orig)-w+1)])
    axes[0, col].plot(np.arange(w//2, len(orig)-w//2+1), rm, 'k-', lw=2, label='Rolling mean')
    axes[0, col].legend(fontsize=8)

    axes[1, col].plot(diffed, color='navy', lw=0.9)
    axes[1, col].set_title(f'After 1st Difference')
    axes[1, col].set_ylabel('Delta')
    # Rolling mean of differenced
    rm_d = np.array([diffed[i:i+w].mean() for i in range(len(diffed)-w+1)])
    axes[1, col].plot(np.arange(w//2, len(diffed)-w//2+1), rm_d, 'k-', lw=2)

plt.suptitle('Time Series Stationarity: Original vs Differenced', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('ts_stationarity.png', dpi=80, bbox_inches='tight')
plt.show()
print('Level 2 complete.')

## Real-World Example 1: AR Forecasting on Synthetic Sales Data

Fit an AR(p) model via OLS, forecast the next 10 steps, and compute MAE/RMSE on a held-out temporal test window.

In [ ]:
# Cell 8: AR(p) model fitting and forecasting via OLS
# AR(p): x_t = c + phi_1*x_{t-1} + ... + phi_p*x_{t-p} + eps_t
# Equivalent to: build design matrix X where each row is [1, x_{t-1}, ..., x_{t-p}]
# Fit via OLS, forecast recursively

def fit_ar_model(x: np.ndarray, p: int) -> dict:
    """Fit AR(p) model by OLS.
    
    Builds lag matrix and solves linear system.
    Returns dict with coefficients, residuals, fitted values.
    """
    n = len(x)
    # Design matrix: X[t] = [1, x_{t-1}, ..., x_{t-p}]
    X_mat = np.column_stack([np.ones(n - p)] +
                            [x[p - k: n - k] for k in range(1, p + 1)])
    y_vec = x[p:]  # Target: x_t for t=p,...,n-1
    # OLS: coefficients = (X^T X)^{-1} X^T y
    coefs, _, _, _ = np.linalg.lstsq(X_mat, y_vec, rcond=None)
    fitted = X_mat @ coefs
    residuals = y_vec - fitted
    return {'coefs': coefs, 'residuals': residuals, 'fitted': fitted, 'p': p}


def forecast_ar(model: dict, x_history: np.ndarray, h: int) -> np.ndarray:
    """Recursive AR(p) forecast h steps ahead.
    
    Uses last p values of x_history, then extends using own forecasts.
    """
    p = model['p']
    coefs = model['coefs']
    forecasts = np.empty(h)
    buffer = list(x_history[-p:])  # Last p observed values

    for step in range(h):
        # Feature vector: [1, lag_1, ..., lag_p]
        feat = np.array([1.0] + buffer[-p:][::-1])
        pred = coefs @ feat
        forecasts[step] = pred
        buffer.append(pred)  # Use forecast as future history (recursive)

    return forecasts


# Generate synthetic sales data: AR(2) with trend and noise
rng = np.random.default_rng(42)
N_SALES = 200
trend = 0.1 * np.arange(N_SALES)
ar2_noise = simulate_ar1(0.65, 1.0, N_SALES, rng=rng)  # AR(1) as approximation
sales = 50 + trend + ar2_noise * 5

# Temporal split (never shuffle time series!)
TRAIN_END = 180
H_FORECAST = 20  # Forecast 20 steps ahead
train = sales[:TRAIN_END]
test = sales[TRAIN_END:TRAIN_END + H_FORECAST]

# Detrend the training data (remove linear trend before AR fitting)
t_train = np.arange(TRAIN_END)
trend_coefs = np.polyfit(t_train, train, deg=1)
train_detrended = train - np.polyval(trend_coefs, t_train)

# Fit AR(2) model
P_ORDER = 2
model = fit_ar_model(train_detrended, P_ORDER)
print(f'AR({P_ORDER}) coefficients: intercept={model["coefs"][0]:.4f}, '
      f'phi_1={model["coefs"][1]:.4f}, phi_2={model["coefs"][2]:.4f}')
print(f'Residual std: {model["residuals"].std():.4f}')

# Forecast: predict detrended, then add trend back
forecasts_detrended = forecast_ar(model, train_detrended, H_FORECAST)
t_test = np.arange(TRAIN_END, TRAIN_END + H_FORECAST)
trend_future = np.polyval(trend_coefs, t_test)
forecasts = forecasts_detrended + trend_future

# Baseline: naive (last observed value)
naive_forecast = np.full(H_FORECAST, train[-1])
# Baseline: moving average (last 10 values)
ma_forecast = np.full(H_FORECAST, train[-10:].mean())

def compute_metrics(actual, predicted):
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted)**2))
    mape = np.mean(np.abs((actual - predicted) / np.abs(actual + 1e-10))) * 100
    return mae, rmse, mape

ar_mae, ar_rmse, ar_mape = compute_metrics(test, forecasts)
naive_mae, naive_rmse, naive_mape = compute_metrics(test, naive_forecast)
ma_mae, ma_rmse, ma_mape = compute_metrics(test, ma_forecast)

print(f'\n{'Method':<20}  {'MAE':>8}  {'RMSE':>8}  {'MAPE (%)':>10}')
print('-' * 50)
print(f'{'AR(2) + trend':<20}  {ar_mae:>8.3f}  {ar_rmse:>8.3f}  {ar_mape:>10.2f}')
print(f'{'Naive (last val)':<20}  {naive_mae:>8.3f}  {naive_rmse:>8.3f}  {naive_mape:>10.2f}')
print(f'{'Moving Avg (10)':<20}  {ma_mae:>8.3f}  {ma_rmse:>8.3f}  {ma_mape:>10.2f}')

# Plot
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(sales, color='navy', lw=1.2, alpha=0.8, label='True sales')
ax.axvline(TRAIN_END, color='black', ls='--', lw=1.5, label='Train/test split')
ax.plot(t_test, forecasts, 'r-o', lw=2, ms=5, label=f'AR(2) forecast (MAE={ar_mae:.2f})')
ax.plot(t_test, naive_forecast, 'g--', lw=1.5, label=f'Naive (MAE={naive_mae:.2f})')
ax.set_xlabel('Time')
ax.set_ylabel('Sales')
ax.set_title('AR(2) Forecasting: Temporal Split (Never Shuffle!)')
ax.legend()
plt.tight_layout()
plt.savefig('ts_forecast.png', dpi=80, bbox_inches='tight')
plt.show()

## Real-World Example 2: Rolling-Window Temporal Cross-Validation

Implement time series cross-validation using rolling and expanding windows. Never use KFold — it violates temporal ordering.

In [ ]:
# Cell 10: Time series cross-validation — rolling vs expanding window
# Rolling window: fixed-size training set slides forward
# Expanding window: training set grows as more data becomes available

def ts_cross_val(x: np.ndarray, p: int, h: int = 5,
                  n_splits: int = 8, mode: str = 'expanding') -> dict:
    """Time series cross-validation.
    
    Args:
        x: Time series
        p: AR order
        h: Forecast horizon
        n_splits: Number of CV folds
        mode: 'expanding' or 'rolling'
    
    Returns dict with fold errors and aggregate statistics.
    """
    n = len(x)
    min_train = max(p * 4, 30)  # Minimum training length
    # Fold start points spaced evenly through the series
    step = (n - min_train - h) // n_splits
    if step < 1:
        step = 1
    window_size = min_train + step * 2  # For rolling mode

    fold_maes = []
    fold_details = []

    for fold in range(n_splits):
        test_start = min_train + fold * step
        test_end = min(test_start + h, n)
        if test_end > n:
            break

        if mode == 'expanding':
            train_start = 0
        else:  # Rolling: fixed-size window
            train_start = max(0, test_start - window_size)

        x_train = x[train_start:test_start]
        x_test = x[test_start:test_end]

        if len(x_train) < p + 2:
            continue

        model = fit_ar_model(x_train, p)
        preds = forecast_ar(model, x_train, len(x_test))
        mae = np.mean(np.abs(x_test - preds))
        fold_maes.append(mae)
        fold_details.append({
            'fold': fold,
            'train_size': len(x_train),
            'test_start': test_start,
            'mae': mae
        })

    return {
        'fold_maes': fold_maes,
        'mean_mae': np.mean(fold_maes) if fold_maes else float('nan'),
        'std_mae': np.std(fold_maes) if fold_maes else float('nan'),
        'folds': fold_details
    }


# Use stationary AR data for clean comparison
rng = np.random.default_rng(88)
ar2 = simulate_ar1(0.7, 1.0, 300, rng=rng)

print('Time Series Cross-Validation: Expanding vs Rolling Window')
print('(Using AR(2) model with p=2, horizon h=5)\n')

for mode in ['expanding', 'rolling']:
    result = ts_cross_val(ar2, p=2, h=5, n_splits=8, mode=mode)
    print(f'{mode.capitalize()} window:')
    print(f'  Mean MAE: {result["mean_mae"]:.4f} ± {result["std_mae"]:.4f}')
    print(f'  Fold MAEs: {["{:.3f}".format(m) for m in result["fold_maes"]]}')
    for fold in result['folds']:
        print(f'    Fold {fold["fold"]}: train_size={fold["train_size"]}, '
              f'test_start={fold["test_start"]}, MAE={fold["mae"]:.4f}')
    print()

print('=== Why NOT to use sklearn KFold for time series ===')
print('KFold randomly shuffles split indices -> future data enters training set')
print('This leaks information: model "knows" future values, giving optimistic scores')
print('Always maintain temporal ordering: train on past, test on future only')

## Real-World Example 3: Seasonality Decomposition + Method Comparison

Implement additive seasonal decomposition (trend + seasonal + residual) using moving averages, and compare forecasting methods on a seasonal synthetic series.

In [ ]:
# Cell 12: Seasonal decomposition + comprehensive comparison plot

def seasonal_decompose(x: np.ndarray, period: int) -> dict:
    """Additive seasonal decomposition: x_t = trend_t + seasonal_t + residual_t.
    
    Uses centered moving average for trend estimation.
    Seasonal component: average deviation from trend at each seasonal position.
    """
    n = len(x)
    # Step 1: Centered moving average as trend estimate
    half = period // 2
    trend = np.full(n, np.nan)
    for t in range(half, n - half):
        trend[t] = np.mean(x[t - half: t + half + 1])

    # Step 2: Detrend
    detrended = x - trend

    # Step 3: Average seasonal component at each phase
    seasonal_pattern = np.zeros(period)
    for s in range(period):
        indices = [t for t in range(half, n - half) if t % period == s % period]
        if indices:
            seasonal_pattern[s] = np.nanmean(detrended[indices])

    # Center the seasonal component (should sum to 0 over one period)
    seasonal_pattern -= seasonal_pattern.mean()
    seasonal = np.array([seasonal_pattern[t % period] for t in range(n)])
    residual = x - trend - seasonal

    return {'trend': trend, 'seasonal': seasonal, 'residual': residual, 'period': period}


# Generate seasonal + trend time series
rng = np.random.default_rng(42)
N_SEASON = 240  # 20 periods of 12
period = 12
t = np.arange(N_SEASON)
trend_comp = 0.05 * t
seasonal_comp = 5 * np.sin(2 * np.pi * t / period)
noise_comp = rng.normal(0, 1.0, N_SEASON)
x_seasonal = 20 + trend_comp + seasonal_comp + noise_comp

# Decompose
decomp = seasonal_decompose(x_seasonal, period)

# Forecast methods comparison on last 24 steps as test
SPLIT = 216
x_train_s = x_seasonal[:SPLIT]
x_test_s = x_seasonal[SPLIT:]
H_S = len(x_test_s)

# Method 1: Naive seasonal (repeat last full period)
naive_seasonal = np.tile(x_train_s[-period:], H_S // period + 1)[:H_S]

# Method 2: Deseasonalize + AR model + re-add seasonal
decomp_train = seasonal_decompose(x_train_s, period)
# Fit AR(2) on deseasonalized + detrended residual
resid_train = np.where(np.isnan(decomp_train['residual']), 0, decomp_train['residual'])
resid_model = fit_ar_model(resid_train[period:], 2)  # Skip NaN-padded beginning
resid_forecasts = forecast_ar(resid_model, resid_train[period:], H_S)
# Add back trend and seasonal
t_future = np.arange(SPLIT, SPLIT + H_S)
trend_future_s = np.polyval(np.polyfit(np.arange(SPLIT), x_train_s, 1), t_future)
seasonal_future = np.array([decomp_train['seasonal'][t % period] for t in t_future])
ar_seasonal_forecast = trend_future_s + seasonal_future + resid_forecasts

# Method 3: Naive (last value)
naive_last = np.full(H_S, x_train_s[-1])

# Metrics
methods_s = [
    ('Seasonal Naive', naive_seasonal),
    ('AR + Decomp', ar_seasonal_forecast),
    ('Naive (last val)', naive_last),
]

print('Forecasting comparison on seasonal time series:')
print(f'{'Method':<22}  {'MAE':>8}  {'RMSE':>8}  {'MAPE (%)':>10}')
print('-' * 56)
for name, pred in methods_s:
    mae, rmse, mape = compute_metrics(x_test_s, pred)
    print(f'{name:<22}  {mae:>8.3f}  {rmse:>8.3f}  {mape:>10.2f}')

# Visualization
fig = plt.figure(figsize=(15, 10))
gs = GridSpec(3, 2, figure=fig)

# Decomposition
ax_orig = fig.add_subplot(gs[0, 0])
ax_orig.plot(x_seasonal, color='navy', lw=0.9, label='Original')
ax_orig.plot(decomp['trend'], color='red', lw=2, label='Trend')
ax_orig.legend()
ax_orig.set_title('Original + Trend')

ax_seas = fig.add_subplot(gs[1, 0])
ax_seas.plot(decomp['seasonal'], color='green', lw=0.9)
ax_seas.set_title('Seasonal Component')

ax_res = fig.add_subplot(gs[2, 0])
ax_res.plot(decomp['residual'], color='gray', lw=0.9)
ax_res.set_title('Residual')

# Forecasts
ax_fc = fig.add_subplot(gs[:, 1])
ax_fc.plot(x_seasonal, color='navy', lw=1.2, alpha=0.7, label='True')
ax_fc.axvline(SPLIT, color='black', ls='--', lw=1.5, label='Train/test split')
colors_fc = ['firebrick', 'green', 'orange']
for (name, pred), color in zip(methods_s, colors_fc):
    mae_v, _, _ = compute_metrics(x_test_s, pred)
    ax_fc.plot(np.arange(SPLIT, SPLIT + H_S), pred, '-',
               color=color, lw=2, label=f'{name} (MAE={mae_v:.2f})')
ax_fc.set_title('Forecast Comparison (last 24 steps)')
ax_fc.legend(fontsize=9)
ax_fc.set_xlabel('Time')

plt.suptitle('Time Series: Seasonal Decomposition and Forecast Comparison', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('ts_seasonal_comparison.png', dpi=80, bbox_inches='tight')
plt.show()

print('\nKey Takeaways:')
print('- Always test stationarity before modeling (rolling mean/variance, AR coef test)')
print('- Differencing (d=1) converts most trending series to stationary')
print('- NEVER use KFold for time series — use rolling or expanding window CV')
print('- Seasonal data needs seasonal naive as baseline, not just last-value naive')
print('- Decompose first (trend + seasonal), model residuals, then reconstruct forecast')